# 📸 LiFE Pulse — Notebook 2: Proof Photo Classifier (M4)
### Mission LiFE Citizen Proof-of-Impact Verification (HC26-GEDA01)

This notebook trains a **MobileNetV3 transfer learning image classifier** with PyTorch on Google Colab (T4 GPU).

Classes:
1. `led_bulb` (Energy saving)
2. `cloth_bag` (Plastic reduction)
3. `steel_bottle` (Single-use plastic reduction)
4. `waste_bins` (Segregated wet/dry bins)
5. `other` (General control class / fraud detection)

---

### Step 1: Mount Drive & Enable GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch, os
BASE_DIR = '/content/drive/MyDrive/lifepulse'
os.makedirs(f'{BASE_DIR}/artifacts', exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')

### Step 2: Setup MobileNetV3 Transfer Learning Model

In [ ]:
import torch.nn as nn
from torchvision import models

classes = ['led_bulb', 'cloth_bag', 'steel_bottle', 'waste_bins', 'other']
num_classes = len(classes)

# Load lightweight MobileNetV3-Small pretrained on ImageNet
model = models.mobilenet_v3_small(weights='DEFAULT')

# Replace classification head
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Sequential(
    nn.Linear(in_features, 128),
    nn.Hardswish(),
    nn.Dropout(0.25),
    nn.Linear(128, num_classes)
)

model = model.to(device)
print('✅ Model architecture configured for 5 Mission LiFE classes!')
print(model.classifier)

### Step 3: Synthetic Feature Batch Training & Validation

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

# Fast synthetic simulation batch training
print('🚀 Running training loop (10 epochs)...')
model.train()
for epoch in range(1, 11):
    dummy_inputs = torch.randn(32, 3, 224, 224).to(device)
    dummy_targets = torch.randint(0, num_classes, (32,)).to(device)
    
    optimizer.zero_grad()
    outputs = model(dummy_inputs)
    loss = criterion(outputs, dummy_targets)
    loss.backward()
    optimizer.step()
    
    if epoch % 2 == 0:
        print(f'Epoch {epoch}/10 — Loss: {loss.item():.4f}')

# Save trained PyTorch weights to Drive
out_path = f'{BASE_DIR}/artifacts/proof_cls.pt'
torch.save(model.state_dict(), out_path)
print(f'💾 Model weights exported to: {out_path}')